In [ ]:
"""
UTN Equity Indicators (ACS 5-year) -> Single Excel Table (San Joaquin County Tracts)

Output columns (in this exact order):
1) GEOID
2) Senior_65   = Older individuals (65yrs and up)
3) Disability  = People with Disabilities (total, both sexes, all ages)
4) Poverty     = People below poverty level
5) Youth_18    = Youth (17yrs and below)
6) EG          = High Transit Dependent Ethnic Groups (proxy: Total pop - Non-Hispanic White alone)
7) NVH         = Households with No Vehicles
8) FB          = Foreign Born

Reproducible:
- Change ACS_YEAR to update years in the future (e.g., 2024 when available).
- Keep STATE_FIPS/COUNTY_FIPS to switch geographies.
"""

import requests
import pandas as pd

# ============================================================
# 1) SETTINGS (edit these when needed)
# ============================================================
ACS_YEAR = 2023
ACS_DATASET = "acs/acs5"          # 5-year ACS dataset
STATE_FIPS = "06"                # California
COUNTY_FIPS = "077"              # San Joaquin County
CENSUS_API_KEY = ""              # Optional but recommended

OUT_XLSX = f"SanJoaquin_ACS{ACS_YEAR}_5yr_Tracts_UTN_Indicators.xlsx"


# ============================================================
# 2) ACS VARIABLE CODES
# ============================================================

# (a) Seniors 65+ (B01001: Sex by Age) — sum of male+female 65+ bins
SENIOR_CODES = [
    "B01001_020E","B01001_021E","B01001_022E","B01001_023E","B01001_024E","B01001_025E",
    "B01001_044E","B01001_045E","B01001_046E","B01001_047E","B01001_048E","B01001_049E"
]

# (d) Youth 0–17 (≤17) (B01001: Sex by Age) — sum of male+female under-18 bins
YOUTH_CODES = [
    "B01001_003E","B01001_004E","B01001_005E","B01001_006E",
    "B01001_027E","B01001_028E","B01001_029E","B01001_030E"
]

# (b) Disability total (B18101: Sex by Age by Disability Status)
# Sum all "With a disability" cells across male & female age groups
DIS_CODES = [
    "B18101_004E","B18101_007E","B18101_010E","B18101_013E","B18101_016E","B18101_019E",
    "B18101_023E","B18101_026E","B18101_029E","B18101_032E","B18101_035E","B18101_038E"
]

# (c) Poverty below poverty level (B17001: Poverty status in past 12 months)
POV_BELOW = "B17001_002E"  # below poverty level (people)

# (f) Households with no vehicles available (B08201)
NVH = "B08201_002E"

# (g) Foreign born population (B05002)
FB = "B05002_013E"

# (e) Ethnic Groups proxy: All except Non-Hispanic White alone (B03002)
TOTAL_POP = "B03002_001E"
NH_WHITE = "B03002_003E"

# Variables to request from the API
GET_VARS = (
    SENIOR_CODES
    + YOUTH_CODES
    + DIS_CODES
    + [POV_BELOW, NVH, FB, TOTAL_POP, NH_WHITE]
)


# ============================================================
# 3) FUNCTION: Pull tract data from the Census API
# ============================================================
def fetch_acs_tract_data(
    year: int,
    dataset: str,
    state_fips: str,
    county_fips: str,
    variables: list[str],
    api_key: str = "",
) -> pd.DataFrame:
    """
    Fetch ACS tract-level data for a given state/county and list of variables.
    Returns a DataFrame with requested ACS variables + geography columns: state, county, tract.
    """
    url = f"https://api.census.gov/data/{year}/{dataset}"
    params = {
        "get": ",".join(variables),
        "for": "tract:*",
        "in": f"state:{state_fips}+county:{county_fips}",
    }
    if api_key:
        params["key"] = api_key

    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    headers = data[0]
    rows = data[1:]
    return pd.DataFrame(rows, columns=headers)


# ============================================================
# 4) PULL DATA
# ============================================================
df = fetch_acs_tract_data(
    year=ACS_YEAR,
    dataset=ACS_DATASET,
    state_fips=STATE_FIPS,
    county_fips=COUNTY_FIPS,
    variables=GET_VARS,
    api_key=CENSUS_API_KEY,
)

# Convert requested ACS variables to numeric (API returns strings)
for v in GET_VARS:
    df[v] = pd.to_numeric(df[v], errors="coerce")

# Build 11-digit GEOID = state(2) + county(3) + tract(6)
df["GEOID"] = (
    df["state"].astype(str).str.zfill(2)
    + df["county"].astype(str).str.zfill(3)
    + df["tract"].astype(str).str.zfill(6)
)


# ============================================================
# 5) COMPUTE INDICATORS
# ============================================================
df["Senior_65"] = df[SENIOR_CODES].sum(axis=1, skipna=True)
df["Youth_18"] = df[YOUTH_CODES].sum(axis=1, skipna=True)
df["Disability"] = df[DIS_CODES].sum(axis=1, skipna=True)

df["Poverty"] = df[POV_BELOW]
df["NVH"] = df[NVH]
df["FB"] = df[FB]

# EG = Total population - Non-Hispanic White alone (proxy for "all except white")
df["EG"] = (df[TOTAL_POP].fillna(0) - df[NH_WHITE].fillna(0)).clip(lower=0)


# ============================================================
# 6) FINAL OUTPUT (ordered exactly as you requested)
# ============================================================
out = df[
    ["GEOID", "Senior_65", "Disability", "Poverty", "Youth_18", "EG", "NVH", "FB"]
].copy()

# Sort for consistent output
out = out.sort_values("GEOID").reset_index(drop=True)

# Export to Excel
out.to_excel(OUT_XLSX, index=False)

print(f"✅ Saved Excel: {OUT_XLSX}")
print(f"✅ Rows (tracts): {len(out)}")
display(out.head(5))


✅ Saved Excel: SanJoaquin_ACS2023_5yr_Tracts_UTN_Indicators.xlsx
✅ Rows (tracts): 174


,GEOID,Senior_65,Disability,Poverty,Youth_18,EG,NVH,FB
0,06077000101,136,340,712,651,1874,122,885
1,06077000102,344,611,960,258,1863,609,600
2,06077000300,415,324,959,591,2063,241,505
3,06077000401,420,414,620,501,2100,77,542
4,06077000402,579,882,1227,1430,3944,514,615


In [ ]:
from google.colab import files
files.download(OUT_XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>